In [1]:
# DATASET PARA TRABAAJR CON EL PROYECTO:

tareas_sprint = [
    {"id": "T-01", "historia": "Login de usuarios", "puntos": 5, "estado": "Done", "responsable": "Marta"},
    {"id": "T-02", "historia": "Base de datos clientes", "puntos": 8, "estado": "Done", "responsable": "Carlos"},
    {"id": "T-03", "historia": "API de pagos", "puntos": 13, "estado": "In Progress", "responsable": "Marta"},
    {"id": "T-04", "historia": "Dashboard inicial", "puntos": 5, "estado": "done", "responsable": "Lucia"},
    {"id": "T-05", "historia": "Exportar reporte CSV", "puntos": 3, "estado": "To Do", "responsable": "Marta"},
    {"id": "T-06", "historia": "Pruebas de seguridad", "puntos": 5, "estado": "In Progress", "responsable": None},
    {"id": "T-07", "historia": "Documentacion API", "puntos": 2, "estado": "Done", "responsable": "Lucia"},
    {"id": "T-08", "historia": "Optimizacion consultas", "puntos": 8, "estado": "To Do", "responsable": "Marta"},
]

In [2]:
# Limpieza de estados y asignaciones
## Normalizar estados y responsables, y separar las tareas sin responsable

def limpiar_y_detectar_huerfanas(tareas):
    tareas_limpias = []
    tareas_huerfanas = []

    for x in tareas:
        estado_limpio = str(x.get("estado", "")).strip().title()
        responsable_raw = x.get("responsable")

        if responsable_raw is None or str(responsable_raw).strip() == "":
            responsable_limpio = "Sin asignar"
            tarea_normalizada = {
                "id": x["id"],
                "historia": x["historia"],
                "puntos": x["puntos"],
                "estado": estado_limpio,
                "responsable": responsable_limpio,
            }
            tareas_huerfanas.append(tarea_normalizada)
        else:
            responsable_limpio = str(responsable_raw).strip().title()
            tarea_normalizada = {
                "id": x["id"],
                "historia": x["historia"],
                "puntos": x["puntos"],
                "estado": estado_limpio,
                "responsable": responsable_limpio,
            }

        tareas_limpias.append(tarea_normalizada)

    return tareas_limpias, tareas_huerfanas


tareas_limpias, tareas_huerfanas = limpiar_y_detectar_huerfanas(tareas_sprint)
print("Tareas limpias:", len(tareas_limpias), "| Tareas huérfanas:", tareas_huerfanas)

Tareas limpias: 8 | Tareas huérfanas: [{'id': 'T-06', 'historia': 'Pruebas de seguridad', 'puntos': 5, 'estado': 'In Progress', 'responsable': 'Sin asignar'}]


In [5]:
# -------- VELOCIDAD DE SPRINT --------

def calcular_velocidad_sprint(tareas):
    puntos_totales = 0
    puntos_completados = 0

    for x in tareas:
        puntos = x["puntos"]
        puntos_totales += puntos
        if x["estado"] == "Done":
            puntos_completados += puntos

    porcentaje_avance = (
        round((puntos_completados / puntos_totales) * 100, 1)
        if puntos_totales > 0
        else 0.0
    )

    return {
        "puntos_totales": puntos_totales,
        "puntos_completados": puntos_completados,
        "porcentaje_avance": porcentaje_avance,
    }


velocidad = calcular_velocidad_sprint(tareas_limpias)
print(
    f"Velocidad: {velocidad['puntos_completados']}/{velocidad['puntos_totales']} pts "
    f"({velocidad['porcentaje_avance']}%)"
)




Velocidad: 20/49 pts (40.8%)


In [4]:
#----CARGA POR RESPONSABLE----

def auditar_carga_equipo(tareas, umbral_alerta_pct=40.0):
    puntos_por_persona = {}
    total_puntos = 0

    for x in tareas:
        persona = x["responsable"]
        puntos = x["puntos"]
        puntos_por_persona[persona] = puntos_por_persona.get(persona, 0) + puntos
        total_puntos += puntos

    auditoria = {}
    alertas = []

    for x in puntos_por_persona.items():
        persona, puntos = x
        pct_carga = round((puntos / total_puntos) * 100, 1) if total_puntos > 0 else 0.0
        supera_umbral = pct_carga > umbral_alerta_pct

        auditoria[persona] = {
            "puntos": puntos,
            "porcentaje": pct_carga,
            "alerta": supera_umbral,
        }

        if supera_umbral:
            alertas.append(f"{persona} concentra el {pct_carga}% ({puntos}/{total_puntos} pts)")

    return auditoria, alertas


auditoria, alertas = auditar_carga_equipo(tareas_limpias, umbral_alerta_pct=40.0)
print("Auditoría de carga:", auditoria, "| Alertas:", alertas)

Auditoría de carga: {'Marta': {'puntos': 29, 'porcentaje': 59.2, 'alerta': True}, 'Carlos': {'puntos': 8, 'porcentaje': 16.3, 'alerta': False}, 'Lucia': {'puntos': 7, 'porcentaje': 14.3, 'alerta': False}, 'Sin asignar': {'puntos': 5, 'porcentaje': 10.2, 'alerta': False}} | Alertas: ['Marta concentra el 59.2% (29/49 pts)']
